In [1]:
# !pip install -U stable-baselines3

In [2]:
import os
import numpy as np
import pandas as pd
import pickle as pkl
from datetime import datetime
import pytz

from model import Model
from data_handler import process_data, fetch_data, DataHandler, InferenceDataHandler

In [3]:
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)
ticker = 'TSLA'

# end_date = datetime.now(pytz.UTC).strftime('%Y-%m-%d')
end_date = '2024-10-11'
start_date = end_date
print(start_date, end_date)

2024-10-11 2024-10-11


In [4]:
import time

timeframe = 390
start_t = time.perf_counter()
processed_data = DataHandler(data_dir=data_dir).get_data(ticker=ticker, start_date=start_date, end_date=end_date)[0]
end_t = time.perf_counter()

print(end_t - start_t)
# processed_train_data.describe()

File Data\TSLA_2024-10-11_2024-10-11.csv already exists, skipping download
0.00808669999241829


In [5]:
processed_data.describe()

,bid_price_1,bid_price_2,bid_price_3,bid_price_4,bid_price_5,bid_size_1,bid_size_2,bid_size_3,bid_size_4,bid_size_5,...,processed_ask_price_1,processed_ask_price_2,processed_ask_price_3,processed_ask_price_4,processed_ask_price_5,processed_ask_size_1,processed_ask_size_2,processed_ask_size_3,processed_ask_size_4,processed_ask_size_5
count,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,391.000000,...,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02,3.910000e+02
mean,219.770563,219.770256,219.769156,219.768210,219.767494,133.222506,113.690537,129.314578,130.138107,122.294118,...,1.633703e-14,9.849467e-15,1.615531e-14,1.542841e-14,2.435108e-14,9.086224e-18,-1.362934e-17,-2.725867e-17,-1.362934e-17,2.725867e-17
std,1.016542,1.016569,1.016253,1.016033,1.015887,258.059612,135.001306,217.985840,190.766608,153.779539,...,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00,1.001281e+00
min,216.310000,216.310000,216.300000,216.300000,216.300000,1.000000,1.000000,1.000000,1.000000,1.000000,...,-4.618691e+00,-4.621773e+00,-4.623457e+00,-4.624408e+00,-4.623891e+00,-4.488706e-01,-4.347725e-01,-4.271445e-01,-4.272261e-01,-4.222219e-01
25%,219.085000,219.085000,219.080000,219.080000,219.080000,41.000000,34.500000,40.000000,35.000000,39.000000,...,-5.538290e-01,-5.545764e-01,-5.554295e-01,-5.560897e-01,-5.565523e-01,-3.955001e-01,-3.957169e-01,-3.809675e-01,-3.811653e-01,-3.715151e-01
50%,219.800000,219.800000,219.790000,219.790000,219.790000,100.000000,100.000000,100.000000,100.000000,100.000000,...,1.267557e-01,1.263991e-01,1.256851e-01,1.250737e-01,1.244471e-01,-3.260177e-01,-3.316657e-01,-3.244243e-01,-3.106642e-01,-3.008877e-01
75%,220.360000,220.360000,220.360000,220.360000,220.360000,127.000000,125.000000,130.000000,138.500000,155.000000,...,6.022327e-01,6.021491e-01,6.015323e-01,6.009549e-01,6.002138e-01,-8.081551e-02,-5.879718e-02,-8.505820e-02,-9.352069e-02,-9.489121e-02
max,223.320000,223.320000,223.320000,223.320000,223.320000,2882.000000,1430.000000,2800.000000,2300.000000,2495.000000,...,3.352541e+00,3.354036e+00,3.353982e+00,3.353602e+00,3.352198e+00,8.522409e+00,1.015607e+01,9.155983e+00,9.073507e+00,8.729456e+00


In [6]:

model_dir = 'Models'
# os.makedirs(model_dir, exist_ok=True)
logging_dir = 'logs'

trial = "_env6_trial3_totpen_multi_env_exp_rew_price_sde_false_allinv"

n_env = 8

sac_model = Model(model_dir=model_dir, logging_dir=logging_dir,n_env=n_env)

env_params = {
    "preferred_timeframe" : 390,
    "inventory": 10000,
    "inference": True
}


model_name = f"AAPL_trial{trial}.pt"



In [7]:
env_params["preferred_timeframe"] = timeframe
env_params["inventory"]=1000
start_t = time.perf_counter()
rew, trades = sac_model.test(processed_data, model=None, model_name=model_name, env_params=env_params)
end_t = time.perf_counter()
print(" Reward: ", rew)
print(" Time taken: ", end_t - start_t)
trades.to_csv(f"{ticker}_trades.csv")
trades

Using cuda device


c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
c:\Users\yashv\Documents\Companies\blockhouse\ml_paper_replicate\Blockhouse-ML\backtest_lib\model.py:187: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recom

Model loaded from Models\AAPL_trial_env6_trial3_totpen_multi_env_exp_rew_price_sde_false_allinv.pt
volatilities:  Series([], Name: bid_price_1, dtype: float64)
volatilities:  Series([], Name: bid_price_1, dtype: float64)
volatilities:  Series([], Name: bid_price_1, dtype: float64)
volatilities:  Series([], Name: bid_price_1, dtype: float64)
 Reward:  0
 Time taken:  33.15138110000407


,inventory,step,timestamp,time_elapsed,time_slice,shares,order_type,price
0,879.0,3,2024-10-11 13:33:00,3,3,121.0,market,NaN
1,873.0,20,2024-10-11 13:50:00,20,17,6.0,limit,221.05
2,787.0,40,2024-10-11 14:10:00,40,20,86.0,market,NaN
3,771.0,77,2024-10-11 14:47:00,77,37,16.0,market,NaN
4,760.0,115,2024-10-11 15:25:00,115,38,11.0,market,NaN
5,680.0,148,2024-10-11 15:58:00,148,33,80.0,market,NaN
6,657.0,185,2024-10-11 16:35:00,185,37,23.0,market,NaN
7,551.0,223,2024-10-11 17:13:00,223,38,106.0,market,NaN
8,445.0,254,2024-10-11 17:44:00,254,31,106.0,market,NaN
9,266.0,259,2024-10-11 17:49:00,259,5,179.0,market,NaN
